1. Criando o Ambiente Customizado (O Simulador)

Primeiro, precisamos empacotar a sua equação de regressão linear em uma classe que o agente de RL consiga entender. O agente precisa de três coisas a cada passo: ler o estado, aplicar uma ação e receber uma recompensa.

Found existing installation: gymnasium 1.2.3
Uninstalling gymnasium-1.2.3:
  Would remove:
    /home/klauss/miniconda3/lib/python3.13/site-packages/gymnasium-1.2.3.dist-info/*
    /home/klauss/miniconda3/lib/python3.13/site-packages/gymnasium/*
Proceed (Y/n)? 

In [6]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class AmbienteControleTemperatura(gym.Env):
    def __init__(self):
        super(AmbienteControleTemperatura, self).__init__()
        
        # 1. Espaço de Ação (Actuator): 
        # Supondo um atuador que aceita potência de -1.0 (resfriar máximo) a 1.0 (aquecer máximo).
        # Se for só ligar/desligar, usaríamos spaces.Discrete(2)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        
        # 2. Espaço de Observação (Sensors + Input): 
        # O agente precisa saber a [Temperatura Atual, Setpoint Desejado]
        self.observation_space = spaces.Box(low=0.0, high=100.0, shape=(2,), dtype=np.float32)
        
        # Estado inicial
        self.setpoint = 25.0
        self.temp_atual = 20.0
        
    def step(self, action):
        u = action[0] # Ação tomada pelo agente
        
        # --- AQUI ENTRA A SUA REGRESSÃO LINEAR ---
        # Exemplo: T(k+1) = a*T(k) + b*u(k) + c
        # Substitua a, b e c pelos coeficientes que você identificou!
        a, b, c = 0.95, 1.2, 0.1 
        
        # Atualiza a temperatura com base no modelo do sistema
        self.temp_atual = a * self.temp_atual + b * u + c
        
        # Adicionando um leve ruído (opcional, mas recomendado)
        # Como dados de sensores reais costumam ter variações que o modelo ideal 
        # não captura perfeitamente, o ruído ajuda o agente a ser mais robusto.
        self.temp_atual += np.random.normal(0, 0.1)
        
        # --- FUNÇÃO DE RECOMPENSA ---
        # O agente é punido com base no quão longe está do Setpoint
        erro = abs(self.setpoint - self.temp_atual)
        reward = -erro 
        
        # Penalidade extra opcional: punir variações bruscas no atuador para economizar energia
        # reward -= 0.1 * (u ** 2)
        
        # Condições de término (se a temperatura explodir, encerra o episódio)
        terminated = bool(self.temp_atual > 100.0 or self.temp_atual < 0.0)
        truncated = False
        
        info = {}
        observacao = np.array([self.temp_atual, self.setpoint], dtype=np.float32)
        
        return observacao, reward, terminated, truncated, info

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Inicia um novo episódio com temperaturas e setpoints aleatórios
        # para o agente aprender a controlar em qualquer situação.
        self.temp_atual = np.random.uniform(15.0, 30.0)
        self.setpoint = np.random.uniform(20.0, 28.0)
        
        observacao = np.array([self.temp_atual, self.setpoint], dtype=np.float32)
        return observacao, {}

2. Treinando o Agente (O Controlador)

Com o simulador pronto, usamos o PPO (Proximal Policy Optimization), que é um dos melhores algoritmos para controle contínuo.

In [ ]:
from stable_baselines3 import PPO

# 1. Instancia o ambiente simulado
env = AmbienteControleTemperatura()

# 2. Define o modelo de RL (Rede Neural Multi-Layer Perceptron)
# verbose=1 vai imprimir o progresso no terminal
modelo_controlador = PPO("MlpPolicy", env, verbose=1, learning_rate=0.001)

print("Iniciando treinamento do Controlador...")
# 3. Treina o agente por um número X de passos no simulador
# Como o ambiente é uma equação simples, isso rodará em segundos.
modelo_controlador.learn(total_timesteps=100000)

# 4. Salva o "cérebro" treinado
modelo_controlador.save("controlador_temperatura_ppo")
print("Treinamento concluído e modelo salvo!")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/home/klauss/miniconda3/lib/python3.13/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Iniciando treinamento do Controlador...
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 83.5      |
|    ep_rew_mean     | -1.46e+03 |
| time/              |           |
|    fps             | 1501      |
|    iterations      | 1         |
|    time_elapsed    | 1         |
|    total_timesteps | 2048      |
----------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 78.4        |
|    ep_rew_mean          | -1.31e+03   |
| time/                   |             |
|    fps                  | 1146        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009733107 |
|    clip_fraction        | 0.0956      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.4        |
|    explained_varianc

3. Implementação na Prática (Malha Fechada Real)

Depois de treinado, você carrega esse modelo e o coloca dentro do seu loop de controle real (por exemplo, em um script Python que se comunica com seu hardware via porta serial para ler o sensor e enviar o comando PWM para o atuador). O modelo treinado substitui aquele bloco vermelho da sua imagem.

In [ ]:
# Carrega o modelo treinado
modelo = PPO.load("controlador_temperatura_ppo")

# Loop principal de controle (rodando em tempo real)
while True:
    # 1. Lê os dados do sensor físico real
    temperatura_medida = ler_sensor_temperatura() 
    setpoint_desejado = ler_setpoint_usuario()
    
    # Monta a observação
    obs = np.array([temperatura_medida, setpoint_desejado], dtype=np.float32)
    
    # 2. O Controlador de ML decide a ação baseada no estado
    # deterministic=True garante que ele não tome ações exploratórias/aleatórias em produção
    acao, _estados = modelo.predict(obs, deterministic=True) 
    comando_atuador = acao[0]
    
    # 3. Envia para a planta
    enviar_comando_para_atuador(comando_atuador)
    
    # Aguarda o próximo ciclo de amostragem (ex: 1 segundo)
    time.sleep(1)